# 🤖 AutoFix-Agents: Multi-Agent Self-Correcting Bug Fixer

*Planner → Executor (tool-using ReAct loop) → Critic (hidden tests) → Reflector → Memory*

---

# 🤖 AutoFix-Agents: A Multi-Agent, Self-Correcting Bug-Fixing System

An **agentic AI** project where three cooperating agents (Planner, Executor, Critic) autonomously
diagnose and fix bugs in Python code, using real tool calls, episodic memory, reflection, and a
**hidden-test evaluation harness** that catches agents which only "game" the visible tests.

Everything lives in one notebook: `agentic_autofix.ipynb`.

## Architecture

```
                 ┌────────────┐   plan + memory hints
 bug report ───▶ │  PLANNER   │────────────────────────┐
                 └────────────┘                        ▼
        ┌───────────────────────────────────────────────────────┐
        │ EXECUTOR (ReAct loop with tool use)                   │
        │  list_files · read_file · write_file · run_tests      │
        │  search_memory · finish                               │
        └───────────────┬───────────────────────────────────────┘
                        │ submission
                        ▼
                 ┌────────────┐  fail   ┌────────────┐
                 │   CRITIC   │────────▶│ REFLECTOR  │──▶ feedback ─▶ back to EXECUTOR
                 │ visible +  │         └────────────┘
                 │ hidden tests│ pass
                 └─────┬──────┘
                       ▼
              store lesson in MEMORY
```

## Project structure

```
agentic_autofix.ipynb   # the full project (code + comments)
README.md               # this file
```

## Setup

```bash
pip install jupyter anthropic      # anthropic only needed for LIVE mode
export ANTHROPIC_API_KEY=sk-ant-...   # optional
jupyter notebook agentic_autofix.ipynb
```

Python 3.9+ is required. No other dependencies.

## Modes

- **LIVE**: used automatically when `ANTHROPIC_API_KEY` is set. Real Claude agents solve the tasks.
- **DEMO**: used otherwise. A scripted stand-in replays agent behaviour so you can verify the whole
  pipeline (tools, critic, reflection loop, metrics) offline and for free. DEMO results are **not**
  a measure of model capability.

Override with `FORCE_DEMO = True` or the `AGENT_MODEL` environment variable.

## Benchmark tasks

| ID | Bug | Subtlety |
|---|---|---|
| `binary_search` | Loop bound off by one | Fails only on boundary/single-element inputs |
| `mutable_default` | Shared mutable default argument | State leaks between calls |
| `lru_cache` | Reads don't update recency | Wrong eviction order |
| `merge_intervals` | Merged end overwritten, not maxed | Nested intervals |
| `token_bucket` | Two bugs (`>` vs `>=`, uncapped refill) | Visible tests catch only one; hidden tests catch the other |

## Reading the results

The final table reports, per task: `solved` (passes visible **and** hidden tests), `rounds`
(Critic rejections + 1), LLM `steps`, `tools` calls, token usage, and wall-clock seconds.
`token_bucket` is designed to trigger the Critic → Reflector → retry path.

## Key design decisions

- **Hidden tests as ground truth.** Visible tests pass is not the same as correct. The Critic's
  verdict is independent of the Executor's own `run_tests` calls.
- **Feedback is the failure report, not the hidden test source.** The Executor sees tracebacks, not the tests.
- **Provider-agnostic LLM interface.** `LLM.chat(system, messages, tools)` returns normalized blocks,
  so another provider is a ~20 line adapter.

## Limitations

- The sandbox is a subprocess with timeouts, **not** a container. Do not run untrusted code with it.
- The benchmark is small (5 tasks) and single-file; it is a framework demo, not a SWE-bench result.
- TF-IDF memory is lexical; embeddings would retrieve better on paraphrased bugs.

## Roadmap / extensions

1. Docker or `nsjail` sandbox for real isolation.
2. Multi-file repos and a `grep`/`run_command` tool; try SWE-bench-Lite instances.
3. Parallel Executors with self-consistency voting, Critic picks the best patch.
4. Embedding-based memory and a learned router that picks cheap vs strong models per step.
5. Ablations: memory on/off, reflection on/off, planner on/off (see the ablation cell).

## Resume bullet (fill in your own measured numbers)

> Built a Planner/Executor/Critic multi-agent system with tool use, episodic memory, and
> reflection; evaluated with hidden-test harness to detect test-overfitting, achieving **X/Y**
> solve rate across **N** tasks with **Z** average tool calls.


## 1. Setup & configuration

In [ ]:
# ===== 1. SETUP & CONFIG =====
import os, sys, re, json, time, math, shutil, difflib, tempfile, subprocess
from dataclasses import dataclass, field
from pathlib import Path
from collections import Counter

# LIVE mode: real Claude agents. Needs `pip install anthropic` + ANTHROPIC_API_KEY.
# DEMO mode: scripted stand-in so the whole pipeline can be tested offline for free.
FORCE_DEMO = False
MODE = "LIVE" if (os.getenv("ANTHROPIC_API_KEY") and not FORCE_DEMO) else "DEMO"
if MODE == "LIVE":
    try:
        import anthropic  # noqa: F401
    except ImportError:
        print("anthropic package not installed -> falling back to DEMO mode")
        MODE = "DEMO"

MODEL = os.getenv("AGENT_MODEL", "claude-sonnet-5-5")  # any tool-use capable Claude model
MAX_STEPS_PER_ROUND = 10   # hard cap on ReAct iterations per executor round (prevents infinite loops)
MAX_ROUNDS = 3             # how many times the Critic may send the Executor back to retry
VERBOSE = True             # print a compact live trace

print(f"Mode: {MODE} | Model: {MODEL if MODE == 'LIVE' else 'scripted demo'}")

## 2. Benchmark: buggy programs, visible tests, hidden tests

Each task has:
- `source`: the buggy `solution.py`
- `tests`: **visible** tests the Executor can run
- `hidden_tests`: used **only by the Critic** to catch overfitting
- `fixes`: reference solutions, used **only by DEMO mode** (one entry per round; the last is fully correct)

In [ ]:
# ===== 2. BENCHMARK TASKS =====
@dataclass
class Task:
    id: str
    title: str
    description: str      # the "bug report" shown to the agents
    source: str           # buggy solution.py
    tests: str            # visible tests
    hidden_tests: str     # critic-only tests
    fixes: list           # DEMO mode only: per-round replacement files

TASKS = []

# ---- Task 1: off-by-one in binary search -------------------------------------------------
TASKS.append(Task(
    id="binary_search", title="Binary search misses boundary elements",
    description="binary_search(arr, target) sometimes returns -1 even though target is in the sorted list.",
    source="""def binary_search(arr, target):
    # Return index of target in sorted list arr, or -1 if absent.
    lo, hi = 0, len(arr) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1
""",
    tests="""import unittest
from solution import binary_search

class T(unittest.TestCase):
    def test_middle(self):
        self.assertEqual(binary_search([1, 3, 5, 7, 9], 5), 2)
    def test_last(self):
        self.assertEqual(binary_search([1, 3, 5, 7, 9], 9), 4)
    def test_single(self):
        self.assertEqual(binary_search([5], 5), 0)
    def test_missing(self):
        self.assertEqual(binary_search([1, 3, 5], 4), -1)
""",
    hidden_tests="""import unittest
from solution import binary_search

class H(unittest.TestCase):
    def test_every_index(self):
        arr = [2, 4, 6, 8, 10, 12]
        for i, v in enumerate(arr):
            self.assertEqual(binary_search(arr, v), i)
    def test_missing_and_empty(self):
        self.assertEqual(binary_search([2, 4, 6], 7), -1)
        self.assertEqual(binary_search([2, 4, 6], 1), -1)
        self.assertEqual(binary_search([], 3), -1)
""",
    fixes=["""def binary_search(arr, target):
    # Return index of target in sorted list arr, or -1 if absent.
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1
"""]))

# ---- Task 2: mutable default argument ----------------------------------------------------
TASKS.append(Task(
    id="mutable_default", title="Tags leak between unrelated calls",
    description="add_tag('a') then add_tag('b') returns ['a', 'b'] on the second call; each call should start fresh.",
    source="""def add_tag(item, tags=[]):
    # Append item to tags and return the list.
    tags.append(item)
    return tags
""",
    tests="""import unittest
from solution import add_tag

class T(unittest.TestCase):
    def test_independent_calls(self):
        self.assertEqual(add_tag("a"), ["a"])
        self.assertEqual(add_tag("b"), ["b"])
""",
    hidden_tests="""import unittest
from solution import add_tag

class H(unittest.TestCase):
    def test_explicit_list_is_extended(self):
        self.assertEqual(add_tag("x", ["y"]), ["y", "x"])
    def test_default_still_fresh_after_explicit(self):
        add_tag("x", ["y"])
        self.assertEqual(add_tag("z"), ["z"])
    def test_explicit_list_mutated_in_place(self):
        base = ["a"]
        add_tag("b", base)
        self.assertEqual(base, ["a", "b"])
""",
    fixes=["""def add_tag(item, tags=None):
    # Append item to tags and return the list.
    if tags is None:
        tags = []
    tags.append(item)
    return tags
"""]))

# ---- Task 3: LRU cache ignores recency on reads ------------------------------------------
TASKS.append(Task(
    id="lru_cache", title="LRU cache evicts the wrong key",
    description="LRUCache evicts a key that was just read. Reads should count as 'recently used'.",
    source="""from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return -1
        return self.data[key]

    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)
""",
    tests="""import unittest
from solution import LRUCache

class T(unittest.TestCase):
    def test_read_refreshes_recency(self):
        c = LRUCache(2)
        c.put(1, 1); c.put(2, 2)
        self.assertEqual(c.get(1), 1)
        c.put(3, 3)
        self.assertEqual(c.get(2), -1)
        self.assertEqual(c.get(1), 1)
        self.assertEqual(c.get(3), 3)
""",
    hidden_tests="""import unittest
from solution import LRUCache

class H(unittest.TestCase):
    def test_capacity_one(self):
        c = LRUCache(1)
        c.put(1, 1); c.put(2, 2)
        self.assertEqual(c.get(1), -1)
        self.assertEqual(c.get(2), 2)
    def test_update_existing_key(self):
        c = LRUCache(2)
        c.put(1, 1); c.put(1, 10)
        self.assertEqual(c.get(1), 10)
    def test_update_refreshes_recency(self):
        c = LRUCache(2)
        c.put(1, 1); c.put(2, 2); c.put(1, 11); c.put(3, 3)
        self.assertEqual(c.get(2), -1)
        self.assertEqual(c.get(1), 11)
""",
    fixes=["""from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)   # a read makes the key most-recently-used
        return self.data[key]

    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)
"""]))

# ---- Task 4: interval merging --------------------------------------------------------------
TASKS.append(Task(
    id="merge_intervals", title="Nested intervals shrink when merged",
    description="merge_intervals([[1,10],[2,3]]) returns [[1,3]] but should return [[1,10]].",
    source="""def merge_intervals(intervals):
    # Merge overlapping [start, end] intervals; return sorted, merged list.
    out = []
    for s, e in sorted(intervals):
        if out and s <= out[-1][1]:
            out[-1][1] = e
        else:
            out.append([s, e])
    return out
""",
    tests="""import unittest
from solution import merge_intervals

class T(unittest.TestCase):
    def test_basic(self):
        self.assertEqual(merge_intervals([[1, 3], [2, 6], [8, 10]]), [[1, 6], [8, 10]])
    def test_nested(self):
        self.assertEqual(merge_intervals([[1, 10], [2, 3]]), [[1, 10]])
""",
    hidden_tests="""import unittest
from solution import merge_intervals

class H(unittest.TestCase):
    def test_touching(self):
        self.assertEqual(merge_intervals([[1, 4], [4, 5]]), [[1, 5]])
    def test_unsorted_input(self):
        self.assertEqual(merge_intervals([[5, 6], [1, 2]]), [[1, 2], [5, 6]])
    def test_empty(self):
        self.assertEqual(merge_intervals([]), [])
    def test_chain_with_nested(self):
        self.assertEqual(merge_intervals([[1, 4], [2, 3], [3, 9], [5, 6]]), [[1, 9]])
""",
    fixes=["""def merge_intervals(intervals):
    # Merge overlapping [start, end] intervals; return sorted, merged list.
    out = []
    for s, e in sorted(intervals):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)   # keep the larger end
        else:
            out.append([s, e])
    return out
"""]))

# ---- Task 5: token bucket (TWO bugs; visible tests expose only one) ---------------------
TOKEN_BUCKET_SRC = """class TokenBucket:
    # Rate limiter. now is an injected clock (seconds) so behaviour is deterministic.
    def __init__(self, capacity, refill_rate, now=0.0):
        self.capacity = capacity
        self.tokens = capacity
        self.rate = refill_rate
        self.last = now

    def allow(self, now, cost=1):
        elapsed = now - self.last
        self.tokens = self.tokens + elapsed * self.rate
        self.last = now
        if self.tokens > cost:
            self.tokens -= cost
            return True
        return False
"""
TASKS.append(Task(
    id="token_bucket", title="Rate limiter rejects valid requests and over-allows after idle",
    description="TokenBucket.allow rejects the request that should use the last token, and after a long idle period it allows more than `capacity` requests in a burst.",
    source=TOKEN_BUCKET_SRC,
    tests="""import unittest
from solution import TokenBucket

class T(unittest.TestCase):
    def test_can_spend_full_capacity(self):
        b = TokenBucket(capacity=2, refill_rate=1)
        self.assertTrue(b.allow(0))
        self.assertTrue(b.allow(0))
        self.assertFalse(b.allow(0))
""",
    hidden_tests="""import unittest
from solution import TokenBucket

class H(unittest.TestCase):
    def test_refill_is_capped(self):
        b = TokenBucket(capacity=2, refill_rate=1)
        results = [b.allow(100) for _ in range(3)]
        self.assertEqual(results, [True, True, False])
    def test_partial_refill(self):
        b = TokenBucket(capacity=4, refill_rate=2)
        for _ in range(4):
            self.assertTrue(b.allow(0))
        self.assertTrue(b.allow(1)); self.assertTrue(b.allow(1))
        self.assertFalse(b.allow(1))
    def test_cost_greater_than_one(self):
        b = TokenBucket(capacity=5, refill_rate=1)
        self.assertTrue(b.allow(0, cost=5))
        self.assertFalse(b.allow(0, cost=1))
""",
    fixes=[
        # round 1: DEMO agent fixes only the bug the visible test exposes (overfits)
        TOKEN_BUCKET_SRC.replace("self.tokens > cost", "self.tokens >= cost"),
        # round 2: after Critic + Reflector feedback, fixes both
        TOKEN_BUCKET_SRC.replace("self.tokens > cost", "self.tokens >= cost").replace(
            "self.tokens = self.tokens + elapsed * self.rate",
            "self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)"),
    ]))

print(f"Loaded {len(TASKS)} tasks:", [t.id for t in TASKS])

## 3. LLM abstraction layer

Agents only talk to `llm.chat(system, messages, tools)`. Responses are normalized to plain dicts (`text` / `tool_use` blocks), so swapping providers means writing one small adapter.

`DemoLLM` is a **scripted stand-in** for offline testing; it is *not* a real model.

In [ ]:
# ===== 3. LLM LAYER =====
@dataclass
class LLMResponse:
    content: list              # normalized blocks: {"type": "text"|"tool_use", ...}
    stop_reason: str
    input_tokens: int = 0
    output_tokens: int = 0


class AnthropicLLM:
    """Thin wrapper around the Anthropic Messages API with retry/backoff."""
    def __init__(self, model):
        import anthropic
        self.client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment
        self.model = model

    def chat(self, system, messages, tools=None, max_tokens=2000):
        kwargs = dict(model=self.model, max_tokens=max_tokens, system=system, messages=messages)
        if tools:
            kwargs["tools"] = tools
        for attempt in range(3):               # simple exponential backoff for transient errors
            try:
                r = self.client.messages.create(**kwargs)
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)
        content = []
        for b in r.content:
            if b.type == "text" and b.text.strip():     # API rejects empty text blocks on replay
                content.append({"type": "text", "text": b.text})
            elif b.type == "tool_use":
                content.append({"type": "tool_use", "id": b.id, "name": b.name, "input": b.input})
        return LLMResponse(content, r.stop_reason, r.usage.input_tokens, r.usage.output_tokens)


class DemoLLM:
    """Scripted stand-in that replays plausible agent behaviour (NOT a real model).
    It exists so the tools, critic, reflection loop and metrics can be tested offline."""
    def __init__(self, tasks):
        self.tasks = {t.id: t for t in tasks}
        self._n = 0

    @staticmethod
    def _is_feedback(m):
        if m["role"] != "user" or isinstance(m["content"], str):
            return False
        return any(b.get("type") == "text" and b["text"].startswith("CRITIC_FEEDBACK") for b in m["content"])

    def chat(self, system, messages, tools=None, max_tokens=2000):
        task_id = re.search(r"TASK_ID:(\w+)", messages[0]["content"]).group(1)
        task = self.tasks[task_id]
        if not tools:                                   # planner or reflector call
            if "PLANNER" in system:
                text = ("1. search_memory for related lessons\n2. read solution.py and locate the faulty logic\n"
                        "3. apply a minimal fix and re-run the tests\n4. enumerate edge cases (empty, boundary, "
                        "idle/extreme values) and re-read the code against each\n5. finish")
            else:
                text = ("The visible tests passed, but a hidden edge case failed, so my fix was incomplete. "
                        "I should re-derive every invariant of the function rather than stop at the first failing test.")
            return LLMResponse([{"type": "text", "text": text}], "end_turn", 200, 90)

        fb = [i for i, m in enumerate(messages) if self._is_feedback(m)]
        rnd = len(fb)                                   # how many times the critic rejected us
        step = sum(1 for m in messages[fb[-1] if fb else 0:] if m["role"] == "assistant")
        fix = task.fixes[min(rnd, len(task.fixes) - 1)]
        if rnd == 0:
            script = [("Let me check whether I have seen a similar bug before.", "search_memory", {"query": task.title}),
                      ("Now read the code.", "read_file", {"path": "solution.py"}),
                      ("Found the faulty logic; applying a minimal fix.", "write_file", {"path": "solution.py", "content": fix}),
                      ("Verify with the visible tests.", "run_tests", {}),
                      ("Tests pass; submitting.", "finish", {"summary": f"Fixed: {task.title}"})]
        else:
            script = [("The critic found more problems. Re-deriving all invariants and rewriting.", "write_file",
                       {"path": "solution.py", "content": fix}),
                      ("Re-run the tests.", "run_tests", {}),
                      ("Submitting the revised fix.", "finish", {"summary": f"Revised fix after critic: {task.title}"})]
        thought, name, args = script[min(step, len(script) - 1)]
        self._n += 1
        blocks = [{"type": "text", "text": thought},
                  {"type": "tool_use", "id": f"toolu_demo_{self._n}", "name": name, "input": args}]
        return LLMResponse(blocks, "tool_use", 300 + 40 * len(messages), 60)


def make_llm():
    return AnthropicLLM(MODEL) if MODE == "LIVE" else DemoLLM(TASKS)

## 4. Episodic memory (TF-IDF retrieval, pure Python)

Stores `(problem text, lesson)` pairs. The Planner and the Executor's `search_memory` tool query it. Successful runs add new lessons, so later tasks can benefit from earlier ones.

In [ ]:
# ===== 4. EPISODIC MEMORY =====
def tokenize(text):
    return re.findall(r"[a-z0-9_]+", text.lower())


class EpisodicMemory:
    def __init__(self):
        self.docs = []                          # each: {"text": ..., "lesson": ...}

    def add(self, text, lesson):
        self.docs.append({"text": text, "lesson": lesson})

    def search(self, query, k=2):
        """Return top-k [(score, doc)] by TF-IDF cosine similarity (score > 0 only)."""
        if not self.docs:
            return []
        N = len(self.docs)
        df = Counter()                          # document frequency per term
        for d in self.docs:
            df.update(set(tokenize(d["text"])))

        def vec(tokens):                        # sublinear TF * smoothed IDF
            tf = Counter(tokens)
            return {t: (1 + math.log(c)) * (math.log((N + 1) / (df.get(t, 0) + 1)) + 1) for t, c in tf.items()}

        def cos(a, b):
            dot = sum(a[t] * b.get(t, 0.0) for t in a)
            na = math.sqrt(sum(v * v for v in a.values()))
            nb = math.sqrt(sum(v * v for v in b.values()))
            return dot / (na * nb) if na and nb else 0.0

        q = vec(tokenize(query))
        scored = sorted(((cos(q, vec(tokenize(d["text"]))), d) for d in self.docs), key=lambda x: -x[0])
        return [(s, d) for s, d in scored[:k] if s > 0]


def build_memory():
    """Seed with a few GENERIC engineering lessons (deliberately not task-specific answers)."""
    m = EpisodicMemory()
    m.add("off by one loop bounds inclusive exclusive termination condition index boundary",
          "Re-derive the loop invariant and test the first, last and single-element cases by hand.")
    m.add("state leaking between calls shared default argument global variable mutation",
          "Anything evaluated once at definition time is shared across calls; create fresh state inside the function.")
    m.add("tests pass but behaviour incomplete edge cases empty input capacity limit idle large values",
          "Before finishing, list edge cases (empty, one item, boundary, extreme) and check each against the code.")
    return m


# quick self-test
_m = build_memory()
print(_m.search("my loop ends too early and misses the last index")[:1])

## 5. Tools and the sandbox

The Executor can only act through these tools. Guardrails live **in the tool layer** (not in the prompt), so a misbehaving model physically cannot bypass them:

- paths are confined to the workspace
- only `.py` files can be written and syntax is checked
- **test files are read-only** (no cheating by editing tests)
- subprocess has a timeout and a minimal environment

In [ ]:
# ===== 5. TOOLS & SANDBOX =====
@dataclass
class Stats:
    llm_calls: int = 0
    steps: int = 0
    tool_calls: int = 0
    in_tokens: int = 0
    out_tokens: int = 0

    def add(self, resp):
        self.llm_calls += 1
        self.in_tokens += resp.input_tokens
        self.out_tokens += resp.output_tokens


def run_unittests(ws, module=None, timeout=15):
    """Run unittest in the workspace in a subprocess. Returns (passed: bool, output: str)."""
    cmd = [sys.executable, "-m", "unittest", "-q"] + ([module] if module else [])
    env = {k: v for k, v in os.environ.items() if k in ("PATH", "SYSTEMROOT", "HOME")}  # minimal env: no API keys leak
    env["PYTHONDONTWRITEBYTECODE"] = "1"
    try:
        p = subprocess.run(cmd, cwd=ws, capture_output=True, text=True, timeout=timeout, env=env)
        return p.returncode == 0, (p.stdout + p.stderr).strip()
    except subprocess.TimeoutExpired:
        return False, f"TIMEOUT after {timeout}s (possible infinite loop)"


class Toolbox:
    def __init__(self, ws, memory, stats):
        self.ws = Path(ws).resolve()
        self.memory = memory
        self.stats = stats
        self.summary = ""

    def _safe(self, rel):
        p = (self.ws / rel).resolve()
        if not p.is_relative_to(self.ws):                  # blocks ../../etc/passwd style escapes
            raise PermissionError("path escapes workspace")
        return p

    # ---- the tools the model can call -----------------------------------------------------
    def tool_list_files(self):
        return "\n".join(sorted(p.name for p in self.ws.glob("*.py") if not p.name.startswith("_")))

    def tool_read_file(self, path):
        return self._safe(path).read_text()

    def tool_write_file(self, path, content):
        name = Path(path).name
        if name.startswith("test_") or name.startswith("_hidden"):
            raise PermissionError("test files are read-only")
        if not path.endswith(".py"):
            raise PermissionError("only .py files can be written")
        compile(content, path, "exec")                     # reject syntax errors before touching disk
        self._safe(path).write_text(content)
        return f"wrote {len(content)} chars to {path}"

    def tool_run_tests(self):
        ok, out = run_unittests(self.ws)
        return ("PASS\n" if ok else "FAIL\n") + out[-2500:]

    def tool_search_memory(self, query):
        if self.memory is None:
            return "memory disabled"
        hits = self.memory.search(query)
        return "\n".join(f"- ({s:.2f}) {d['lesson']}" for s, d in hits) or "no relevant lessons"

    def tool_finish(self, summary):
        self.summary = summary
        return "Submission received. The critic will now verify your work."

    # ---- dispatcher: never raises, always returns text the model can react to -------------
    def call(self, name, args):
        self.stats.tool_calls += 1
        fn = getattr(self, f"tool_{name}", None)
        if fn is None:
            return f"ERROR: unknown tool '{name}'"
        try:
            return str(fn(**args))[:4000]
        except Exception as e:
            return f"ERROR: {type(e).__name__}: {e}"


def _schema(props=None, required=None):
    return {"type": "object", "properties": props or {}, "required": required or []}

TOOL_SPECS = [
    {"name": "list_files", "description": "List the Python files in the workspace.", "input_schema": _schema()},
    {"name": "read_file", "description": "Read a file in the workspace.",
     "input_schema": _schema({"path": {"type": "string"}}, ["path"])},
    {"name": "write_file", "description": "Overwrite a .py file with COMPLETE new content. Test files are read-only.",
     "input_schema": _schema({"path": {"type": "string"}, "content": {"type": "string"}}, ["path", "content"])},
    {"name": "run_tests", "description": "Run the visible unit tests.", "input_schema": _schema()},
    {"name": "search_memory", "description": "Search past lessons relevant to a bug description.",
     "input_schema": _schema({"query": {"type": "string"}}, ["query"])},
    {"name": "finish", "description": "Submit your fix for verification. Call once, when done.",
     "input_schema": _schema({"summary": {"type": "string"}}, ["summary"])},
]

## 6. The agents: Planner, Executor, Critic, Reflector

- **Planner**: one LLM call, no tools; turns the bug report + memory hints into a short plan.
- **Executor**: a ReAct loop (think → tool call → observe) bounded by `MAX_STEPS_PER_ROUND`.
- **Critic**: *not* an LLM. It deterministically runs visible **and hidden** tests and measures patch size. Ground truth beats opinion.
- **Reflector**: one LLM call that converts the Critic's failure report into a diagnosis for the next attempt.

In [ ]:
# ===== 6. AGENTS =====
PLANNER_SYSTEM = """You are the PLANNER in a multi-agent bug-fixing system.
Given a bug report, the file list and lessons from past work, write a SHORT numbered plan (max 6 steps) for the EXECUTOR.
Include which edge cases to verify. Do not write code."""

EXECUTOR_SYSTEM = """You are the EXECUTOR in a multi-agent bug-fixing system. Fix the bug using the tools.
Rules:
1. Read the code before editing. Never guess.
2. Make the MINIMAL fix and write the COMPLETE file content.
3. Test files are read-only. Do not try to change them.
4. Passing the visible tests is NOT enough: hidden tests exist. Reason about edge cases
   (empty input, boundaries, single element, extreme values, repeated calls) against the code.
5. After editing, run the tests. When confident, call `finish` exactly once."""

REFLECT_SYSTEM = """You are the REFLECTOR. You see a failed fix and the critic's report.
In 2-3 sentences, diagnose what the executor MISSED (root cause, not symptoms) and what to re-check. Do not write code."""


def short(obj, n=90):
    s = json.dumps(obj) if not isinstance(obj, str) else obj
    return s if len(s) <= n else s[:n] + "…"


def log(trace, kind, msg):
    trace.append((kind, msg))
    if VERBOSE:
        print(f"  [{kind:<7}] {msg[:170].replace(chr(10), ' ⏎ ')}")


def text_of(resp):
    return "\n".join(b["text"] for b in resp.content if b["type"] == "text").strip()


def add_user_text(messages, text):
    """Append text to the conversation without creating two consecutive user turns."""
    last = messages[-1]
    if last["role"] == "user":
        if isinstance(last["content"], str):
            last["content"] = [{"type": "text", "text": last["content"]}]
        last["content"].append({"type": "text", "text": text})   # after any tool_result blocks
    else:
        messages.append({"role": "user", "content": text})


def run_executor(llm, tools, messages, stats, trace, max_steps=MAX_STEPS_PER_ROUND):
    """ReAct loop. Returns True if the agent called `finish`, False if it ran out of steps."""
    for _ in range(max_steps):
        resp = llm.chat(EXECUTOR_SYSTEM, messages, tools=TOOL_SPECS, max_tokens=3000)
        stats.add(resp)
        stats.steps += 1
        content = resp.content or [{"type": "text", "text": "(no output)"}]
        messages.append({"role": "assistant", "content": content})

        for b in content:
            if b["type"] == "text":
                log(trace, "thought", b["text"])
        calls = [b for b in content if b["type"] == "tool_use"]
        if not calls:                                       # model rambled without acting
            add_user_text(messages, "No tool call detected. Use a tool, or call `finish` if you are done.")
            continue

        results, done = [], False
        for c in calls:                                     # every tool_use MUST get a tool_result
            out = tools.call(c["name"], c["input"])
            log(trace, "tool", f"{c['name']}({short(c['input'])}) -> {out}")
            results.append({"type": "tool_result", "tool_use_id": c["id"], "content": out})
            done = done or c["name"] == "finish"
        messages.append({"role": "user", "content": results})
        if done:
            return True
    return False


def critic(task, ws):
    """Deterministic verdict: visible tests + HIDDEN tests + patch-size check."""
    vis_ok, vis_out = run_unittests(ws)
    hidden_file = Path(ws) / "_hidden_test.py"
    hidden_file.write_text(task.hidden_tests)               # exists only for the duration of this call
    try:
        hid_ok, hid_out = run_unittests(ws, module="_hidden_test")
    finally:
        hidden_file.unlink(missing_ok=True)

    current = (Path(ws) / "solution.py").read_text()
    diff = difflib.unified_diff(task.source.splitlines(), current.splitlines(), lineterm="", n=0)
    changed = sum(1 for l in diff if l and l[0] in "+-" and not l.startswith(("+++", "---")))

    report = []
    if not vis_ok:
        report.append("VISIBLE TESTS FAILED:\n" + vis_out[-1200:])
    if not hid_ok:
        # We reveal the failure output (so the agent can learn) but never the hidden test source.
        report.append("HIDDEN EDGE-CASE CHECKS FAILED:\n" + hid_out[-1200:])
    if changed > 30:
        report.append(f"Patch is large ({changed} changed lines); prefer a minimal fix.")
    return {"passed": vis_ok and hid_ok, "visible": vis_ok, "hidden": hid_ok,
            "changed_lines": changed, "report": "\n\n".join(report) or "all checks passed"}

## 7. Orchestrator

Ties everything together: **plan → execute → critic → (reflect → retry)\* → store lesson**.

In [ ]:
# ===== 7. ORCHESTRATOR =====
@dataclass
class Result:
    task_id: str
    solved: bool
    rounds: int
    steps: int
    tool_calls: int
    in_tokens: int
    out_tokens: int
    seconds: float
    trace: list
    final_code: str
    verdict: dict


def solve_task(task, llm, memory, use_memory=True, max_rounds=MAX_ROUNDS):
    t0 = time.time()
    ws = Path(tempfile.mkdtemp(prefix=f"agent_{task.id}_"))     # isolated workspace per task
    (ws / "solution.py").write_text(task.source)
    (ws / "test_solution.py").write_text(task.tests)
    stats, trace = Stats(), []
    tools = Toolbox(ws, memory if use_memory else None, stats)
    print(f"\n=== {task.id}: {task.title} ===")

    # ---- 1) PLAN -----------------------------------------------------------------------
    hints = memory.search(task.description) if use_memory else []
    hint_text = "\n".join(f"- {d['lesson']}" for _, d in hints) or "(none)"
    r = llm.chat(PLANNER_SYSTEM, [{"role": "user", "content":
                 f"TASK_ID:{task.id}\nBug report: {task.description}\nFiles: {tools.tool_list_files()}\n"
                 f"Past lessons:\n{hint_text}"}], max_tokens=600)
    stats.add(r)
    plan = text_of(r)
    log(trace, "plan", plan)

    messages = [{"role": "user", "content":
                 f"TASK_ID:{task.id}\n# Bug report\n{task.description}\n\n# Plan from the planner\n{plan}\n\n"
                 f"# Workspace files\n{tools.tool_list_files()}\n\nFix the bug in solution.py."}]

    # ---- 2) EXECUTE -> CRITIC -> REFLECT loop ----------------------------------------------
    verdict, rounds = None, 0
    for rnd in range(1, max_rounds + 1):
        rounds = rnd
        log(trace, "round", f"----- executor round {rnd} -----")
        run_executor(llm, tools, messages, stats, trace)
        verdict = critic(task, ws)
        log(trace, "critic", f"visible={verdict['visible']} hidden={verdict['hidden']} "
                             f"changed_lines={verdict['changed_lines']}")
        if verdict["passed"] or rnd == max_rounds:
            break
        code_now = (ws / "solution.py").read_text()
        refl = llm.chat(REFLECT_SYSTEM, [{"role": "user", "content":
                        f"TASK_ID:{task.id}\nBug report: {task.description}\nYour final code:\n{code_now}\n"
                        f"Critic report:\n{verdict['report']}"}], max_tokens=500)
        stats.add(refl)
        reflection = text_of(refl)
        log(trace, "reflect", reflection)
        add_user_text(messages, f"CRITIC_FEEDBACK (round {rnd} rejected)\n{verdict['report']}\n\n"
                                f"Reflection: {reflection}\n\nRevise your fix, run the tests, and finish again.")

    # ---- 3) LEARN --------------------------------------------------------------------------
    if verdict["passed"] and use_memory:
        memory.add(f"{task.title} {task.description}", f"{task.title}: {tools.summary}")

    final_code = (ws / "solution.py").read_text()
    shutil.rmtree(ws, ignore_errors=True)
    print(f"  -> {'SOLVED' if verdict['passed'] else 'FAILED'} in {rounds} round(s)")
    return Result(task.id, verdict["passed"], rounds, stats.steps, stats.tool_calls,
                  stats.in_tokens, stats.out_tokens, time.time() - t0, trace, final_code, verdict)


def run_benchmark(tasks, use_memory=True):
    llm, memory = make_llm(), build_memory()        # fresh memory per benchmark run (fair ablations)
    return [solve_task(t, llm, memory, use_memory) for t in tasks]

## 8. Run the benchmark

In [ ]:
# ===== 8. RUN =====
results = run_benchmark(TASKS, use_memory=True)

## 9. Results

In [ ]:
# ===== 9. RESULTS TABLE =====
def report(results):
    head = f"{'task':<17}{'solved':<8}{'rounds':<8}{'steps':<7}{'tools':<7}{'tok_in':<9}{'tok_out':<9}{'sec':<6}"
    print(head + "\n" + "-" * len(head))
    for r in results:
        print(f"{r.task_id:<17}{'yes' if r.solved else 'NO':<8}{r.rounds:<8}{r.steps:<7}{r.tool_calls:<7}"
              f"{r.in_tokens:<9}{r.out_tokens:<9}{r.seconds:<6.1f}")
    n = len(results)
    print("-" * len(head))
    print(f"solve rate: {sum(r.solved for r in results)}/{n} | avg rounds: {sum(r.rounds for r in results)/n:.2f} "
          f"| avg tool calls: {sum(r.tool_calls for r in results)/n:.1f} | total tokens: "
          f"{sum(r.in_tokens + r.out_tokens for r in results)}")
    if MODE == "DEMO":
        print("NOTE: DEMO mode replays scripted behaviour; these numbers verify the pipeline, not model quality.")

report(results)

In [ ]:
# ===== 9b. INSPECT A TRACE (the Critic -> Reflector -> retry path) =====
def show_trace(result):
    print(f"Trace for {result.task_id} (solved={result.solved}):")
    for kind, msg in result.trace:
        print(f"[{kind:<7}] {msg[:300]}")
    print("\nFinal code:\n" + result.final_code)

show_trace(next(r for r in results if r.task_id == "token_bucket"))

## 10. Ablation: does memory help? (optional)

Set `RUN_ABLATION = True`. In LIVE mode this costs extra API calls; in DEMO mode it only checks that the harness works.

In [ ]:
# ===== 10. OPTIONAL ABLATION =====
RUN_ABLATION = False
if RUN_ABLATION:
    VERBOSE = False
    with_mem = run_benchmark(TASKS, use_memory=True)
    no_mem = run_benchmark(TASKS, use_memory=False)
    for name, res in [("memory ON", with_mem), ("memory OFF", no_mem)]:
        print(f"{name:<11} solve {sum(r.solved for r in res)}/{len(res)} | "
              f"avg rounds {sum(r.rounds for r in res)/len(res):.2f} | "
              f"tokens {sum(r.in_tokens + r.out_tokens for r in res)}")
    VERBOSE = True